## Sequential Multi-Agent Tracing Sample

Sample of tracking using OpenTelemetry with Azure Foundry Agent Services

Agent v2:
https://learn.microsoft.com/en-us/azure/ai-foundry/quickstarts/get-started-code?view=foundry&tabs=python%2Cpython2

PrPr documentation
https://github.com/microsoft/agentsv2-preview/tree/main

### Setup Foundry Project Client

In [55]:
from typing import Any, Dict
import os
import json
import functools
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
import azure.ai.projects as projectslib
from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor
from dotenv import load_dotenv

# Replacement for azure.ai.agents.telemetry.trace_function (v1 SDK)
def trace_function(_func=None, *, name=None):
    """Decorator that wraps a function in an OpenTelemetry span."""
    def decorator(func):
        span_name = name or func.__name__
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            tracer = trace.get_tracer(__name__)
            with tracer.start_as_current_span(span_name):
                return func(*args, **kwargs)
        return wrapper
    if _func is not None:
        return decorator(_func)
    return decorator

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [56]:
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


### Enable Azure Monitor Tracking

* get application insights connection string from the foundry project

In [57]:
# Enable Azure Monitor tracing geting the app insight connection string
application_insights_connection_string =project_client.telemetry.get_application_insights_connection_string()
if not application_insights_connection_string:
    print("Application Insights was not enabled for this project.")
    print("Enable it via the 'Tracing' tab in your AI Foundry project page.")
    exit()
configure_azure_monitor(connection_string=application_insights_connection_string)

In [58]:
scenario = "tracking_multiagent_scenario"
# scenario = os.path.basename(__file__)
tracer = trace.get_tracer(__name__)

print("using scenario:", scenario)
print("using tracer:", tracer)

using scenario: tracking_multiagent_scenario
using tracer: <opentelemetry.sdk.trace.Tracer object at 0x116e9d6d0>


### Set up the trace_function

* adding custom function trace to the OpenTelemetry by using decorator

In [59]:
@trace_function()
def fetch_weather(location: str) -> str:
    """
    Fetches the weather information for the specified location.

    :param location (str): The location to fetch weather for.
    :return: Weather information as a JSON string.
    :rtype: str
    """
    mock_weather_data = {"New York": "Sunny, 25°C", "London": "Cloudy, 18°C", "Tokyo": "Rainy, 22°C"}

    # Adding attributes to the current span
    span = trace.get_current_span()
    span.set_attribute("requested_location", location)

    weather = mock_weather_data.get(location, "Weather data not available for this location.")
    weather_json = json.dumps({"weather": weather})
    return weather_json

@trace_function()
def convert_temperature(temperature: str) -> str:
    """
    Converts temperature between Celsius and Fahrenheit.
    
    :param temperature: Temperature string in format "25°C" or "77°F"
    :return: Converted temperature as JSON string
    :rtype: str
    """
    span = trace.get_current_span()
    span.set_attribute("input_temperature", temperature)
    
    try:
        value = float(''.join(filter(str.isdigit, temperature)))
        unit = 'C' if '°C' in temperature else 'F'
        
        if unit == 'C':
            converted = (value * 9/5) + 32
            result = f"{converted:.1f}°F"
        else:
            converted = (value - 32) * 5/9
            result = f"{converted:.1f}°C"
            
        return json.dumps({"converted_temperature": result})
    except Exception as e:
        return json.dumps({"error": f"Failed to convert temperature: {str(e)}"})

In [60]:
def check_for_celsius(text: str) -> bool:
    """Check if text contains a Celsius temperature"""
    return "°C" in text

In [61]:
# Map function name -> Python callable for the v2 function-call dispatch loop
function_dispatch: Dict[str, Any] = {
    "fetch_weather": fetch_weather,
    "convert_temperature": convert_temperature,
}

In [62]:
def run_agent(agent_name: str, user_prompt: str, conversation_id: str) -> str:
    """Send user_prompt to a named agent and resolve function calls in a loop.
    Returns the agent's final output_text."""
    agent_ref = {"name": agent_name, "type": "agent_reference"}

    response = openai_client.responses.create(
        input=user_prompt,
        conversation=conversation_id,
        extra_body={"agent_reference": agent_ref},
        tool_choice="auto",
    )

    while True:
        pending = [item for item in response.output if item.type == "function_call"]
        if not pending:
            return response.output_text

        input_list: ResponseInputParam = []
        for item in pending:
            fn = function_dispatch.get(item.name)
            if fn is None:
                output = json.dumps({"error": f"unknown function {item.name}"})
            else:
                args = json.loads(item.arguments) if item.arguments else {}
                print(f"  ↳ {item.name}({args})")
                output = fn(**args)
            input_list.append(FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output,
            ))

        response = openai_client.responses.create(
            input=input_list,
            previous_response_id=response.id,
            extra_body={"agent_reference": agent_ref},
        )

In [63]:
# v2 FunctionTool definitions (JSON Schema, not callable sets)
weather_tools = [
    FunctionTool(
        name="fetch_weather",
        description="Fetches the weather information for the specified location.",
        parameters={
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The location to fetch weather for."},
            },
            "required": ["location"],
        },
    ),
]

conversion_tools = [
    FunctionTool(
        name="convert_temperature",
        description="Converts temperature between Celsius and Fahrenheit.",
        parameters={
            "type": "object",
            "properties": {
                "temperature": {"type": "string", "description": "Temperature string in format '25°C' or '77°F'"},
            },
            "required": ["temperature"],
        },
    ),
]

print(f"Prepared {len(weather_tools)} weather tools, {len(conversion_tools)} conversion tools")

Prepared 1 weather tools, 1 conversion tools


### Setup agent functions

In [64]:
# (cell kept for compatibility — tool definitions are in the cell above)

In [65]:
# Create two agents with different roles
weather_agent = project_client.agents.create_version(
    agent_name="weather-assistant",
    definition=PromptAgentDefinition(
        model=settings.model_deployment_name,
        instructions="""You are a helpful weather assistant. Follow these steps:
1. When asked about weather, politely acknowledge the request
2. Use the fetch_weather function to get the weather information
3. Present the weather information in a friendly, conversational way
4. If the temperature is in Celsius, report back only the Celsius temperature
Be descriptive and natural in your responses.""",
        tools=weather_tools,
    ),
)
print(f"Created weather agent: {weather_agent.name} (v{weather_agent.version})")

conversion_agent = project_client.agents.create_version(
    agent_name="conversion-assistant",
    definition=PromptAgentDefinition(
        model=settings.model_deployment_name,
        instructions="""You are a helpful temperature conversion assistant. Follow these steps:
1. When asked to convert a temperature, acknowledge the request
2. Extract the temperature value from the previous messages
3. Use the convert_temperature function to perform the conversion
4. Present both temperatures in a clear, friendly way
Be detailed and explain the conversion clearly.""",
        tools=conversion_tools,
    ),
)
print(f"Created conversion agent: {conversion_agent.name} (v{conversion_agent.version})")

Created weather agent: weather-assistant (v2)
Created conversion agent: conversion-assistant (v1)


In [66]:
# Quick test: single weather agent call
conversation = openai_client.conversations.create()
print(f"Created conversation: {conversation.id}")

result = run_agent(
    agent_name=weather_agent.name,
    user_prompt="Can you tell me what the weather is like in New York today? I'd like the temperature in Celsius °C please.",
    conversation_id=conversation.id,
)
print(f"\nWeather agent > {result}")

Created conversation: conv_3995fb5bbe78978a00m07IprID13hhdq1YPsqfqCBTPIfH2qj3
  ↳ fetch_weather({'location': 'New York'})

Weather agent > Sure! The weather in New York today is sunny with a temperature of 25°C. It sounds like a lovely day to be outside! Would you like to know anything else about the weather there?


### Sequential Agent Runs inside Tracing Scope

* Create two agents: weather assistant agent, and temperature conversion agent (Celsius ↔ Fahrenheit)
* First run the weather assistant agent
* Check output contains Celsius, then run the temperature conversion agent subsequently
* Each agent uses its **own conversation** (sharing a conversation causes `BadRequestError` because the second agent sees the first agent's unresolved function calls)
* The weather result is passed as text context to the conversion agent's prompt

In [67]:
with tracer.start_as_current_span(scenario) as parent_span:
    # Single conversation for the weather agent (carries the trace context)
    conv = openai_client.conversations.create()
    parent_span.set_attribute("conversation.id", conv.id)
    print(f"Conversation: {conv.id}")

    # --- Step 1: Weather agent (uses the conversation) ---
    with tracer.start_as_current_span("weather_agent") as weather_span:
        weather_span.set_attribute("agent.name", weather_agent.name)
        weather_span.set_attribute("conversation.id", conv.id)

        print("\n--- Weather Agent ---")
        weather_result = run_agent(
            agent_name=weather_agent.name,
            user_prompt="Can you tell me what the weather is like in New York today? I'd like the temperature in Celsius °C please.",
            conversation_id=conv.id,
        )
        weather_span.set_attribute("agent.output", weather_result[:500])
        print(f"Weather agent > {weather_result}")

    # --- Step 2: Conversion agent (new conversation, linked via parent trace span) ---
    # Cannot share conversation because the first agent's function_call items
    # would cause 'No tool output found' errors for the second agent.
    if check_for_celsius(weather_result):
        with tracer.start_as_current_span("conversion_agent") as conv_span:
            conv2 = openai_client.conversations.create()
            conv_span.set_attribute("agent.name", conversion_agent.name)
            conv_span.set_attribute("conversation.id", conv2.id)
            conv_span.set_attribute("parent_conversation.id", conv.id)

            print("\n--- Conversion Agent ---")
            conversion_result = run_agent(
                agent_name=conversion_agent.name,
                user_prompt=f"The weather report said: \"{weather_result}\"\n\nPlease convert the Celsius temperature to Fahrenheit and explain the conversion.",
                conversation_id=conv2.id,
            )
            conv_span.set_attribute("agent.output", conversion_result[:500])
            print(f"Conversion agent > {conversion_result}")
    else:
        print("\nNo Celsius temperature found — skipping conversion.")

    print("\n✅ Sequential multi-agent run complete.")

Conversation: conv_0d03fe76395bddef00he5AyeObiGKS8VVdlKPXTlJZPJBssxn4

--- Weather Agent ---
  ↳ fetch_weather({'location': 'New York'})
Weather agent > Sure! The weather in New York today is sunny with a temperature of 25°C. It sounds like a lovely day to be outside! Is there anything else you'd like to know about the weather?

--- Conversion Agent ---
  ↳ convert_temperature({'temperature': '25°C'})
Conversion agent > The temperature of 25°C is equivalent to 77.0°F.

To explain the conversion:
- We multiply 25 by 9/5, which equals 45.
- Then, we add 32 to 45, resulting in 77.
So, 25°C is equal to 77°F.

It's a lovely warm day in New York! Would you like any other temperature conversions or weather information?

✅ Sequential multi-agent run complete.
